# BGE-small + FAISS full-corpus baseline (Colab GPU)

Embeds all 511,958 EnterpriseRAG-Bench documents with the pinned `BAAI/bge-small-en-v1.5`
model and scores the development split, like the BM25+ baseline.

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells in order. Expect about 10 minutes of setup and roughly 1–2 hours of embedding.
3. Work is saved to Google Drive in shards, so after a disconnect rerun all cells and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive for resumable storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"  # Corpus Parquet and embedding shards go here.
!mkdir -p {WORK}

## 2. Get the project code


In [ ]:
import os

BRANCH = "hira/dev"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} https://github.com/hira-mian/Agentic-Enterprise-RAG-System.git /content/project
%cd /content/project
!git log -1 --format='%h %s'

In [ ]:
!pip install -q -r requirements.txt

## 3. Build the full corpus (skipped if already on Drive)

Clones the pinned upstream repository and converts it to Parquet (about 5–10 minutes).

In [ ]:
CORPUS = f"{WORK}/documents.parquet"
if not os.path.exists(CORPUS):
    !python -m src.data.corpus --clone --output {CORPUS}
else:
    print("Using existing", CORPUS)

## 4. Embed passages on the GPU

Each document is split into 500-token windows with 50-token overlap (about 1.5 million
passages). Completed shards are skipped when this cell is rerun.

In [ ]:
!python -m src.retrieval.corpus_dense --corpus {CORPUS} --output {WORK}/dense --device cuda --batch-size 256

## 5. Score the development split

In [ ]:
!python -m src.evaluation.run_retrieval --method dense --index {WORK}/dense --device cuda --split development \
    --questions {WORK}/questions.jsonl

In [ ]:
import json

summary = json.load(open("evaluation/results/dense_full_development/summary.json"))
rows = [("Overall", summary["overall"])] + list(summary["question_types"].items())
print(f"{'type':26} {'scored':>6} {'R@5':>6} {'R@10':>6} {'R@20':>6} {'nDCG@10':>8}")
for name, v in rows:
    if v["recall@10"]["mean"] is None:
        continue
    cells = [v[k]["mean"] for k in ("recall@5", "recall@10", "recall@20", "ndcg@10")]
    print(f"{name:26} {v['recall@10']['scored']:>6} " + " ".join(f"{c:>6.3f}" for c in cells))

## 6. Save the results

This copies the result files to Drive (`rag-dense/results`) and downloads a zip.
Commit the `dense_full_development` folder to `evaluation/results/`, or share the
Drive folder.

In [ ]:
!mkdir -p {WORK}/results && cp -r evaluation/results/dense_full_development {WORK}/results/
!cd evaluation/results && zip -qr /content/dense_full_development.zip dense_full_development
from google.colab import files
files.download("/content/dense_full_development.zip")